# query_translation

> The per-backend translation of the typed query expressions (pass-2 Thread 5; stage 4): `NodeQuery`/`EdgeQuery` → parameterized SQLite SQL over the `nodes`/`edges` schema. THIS module is what makes the typed surface portable — the expression is domain- and backend-neutral; every backend tool owns a translation like this one (the ratified stage-4 split: backend owns translation; the adapter stays generic). Pure functions, unit-tested against an in-memory DB with the production schema — no capability runtime needed.

In [ ]:
#| default_exp query_translation

## Translation rules

- **Predicates** compile to `json_extract(properties, '$.<path>')` comparisons with bound parameters (values NEVER interpolated). Property paths are dotted (`payload.document_id`) and validated against `[A-Za-z0-9_]` segments — paths are interpolated into the JSON-path literal, so validation is the injection guard for the path position.
- **`contains` is case-INSENSITIVE by definition** (P12): `lower(col) LIKE '%' || lower(?) || '%' ESCAPE '\'` with `%`/`_`/`\` escaped in the value. Both sides use SQLite's `lower()` (ASCII-range folding) — consistent, but non-ASCII case folding is NOT performed (recorded; a unicode-folding consumer would be the promotion evidence).
- **Id lists** bind as ONE JSON-array parameter via `json_each(?)` — no SQL parameter-count limits at corpus scale (13k+ ids).
- **Relation / endpoint constraints** compile to correlated `EXISTS` subqueries (never JOINs) — no row multiplication, so `count` needs no DISTINCT and rows need no GROUP BY.
- **`SourcePredicate`** translates content-hash-primary (`json_each(sources)`); a locator constraint raises (unsupported — use `find_nodes_by_source`, which does Python-side locator equality; a recurring need here is promotion evidence).
- **Modes**: `count` > `project` > full (mirrors the result DTO contract). Projected node rows always carry `id` (+ `label` projectable structurally; `sources` projectable as parsed ref dicts); edge rows always carry `id`/`source_id`/`target_id` (+ `relation_type` projectable).

In [ ]:
#| export
import json
import re
from typing import Any, List, Optional, Tuple

from cjm_substrate.core.errors import CapabilityInputError
from cjm_context_graph_primitives.query import (
    NodeQuery, EdgeQuery, PropertyPredicate, SourcePredicate, RelationPredicate,
)

In [ ]:
#| export
_PROP_PATH_RE = re.compile(r"^[A-Za-z0-9_]+(\.[A-Za-z0-9_]+)*$")  # Dotted-path guard (paths are interpolated; values are bound)


def _json_path(
    prop: str,  # Property name or dotted path (e.g. "payload.document_id")
) -> str:  # SQL string literal for json_extract's path argument
    """Validate a dotted property path and render the JSON-path literal."""
    if not _PROP_PATH_RE.match(prop or ""):
        raise CapabilityInputError(
            f"Invalid property path {prop!r} (segments must be [A-Za-z0-9_])",
            fields_invalid=["prop"],
        )
    return "'$." + prop + "'"

In [ ]:
#| export
def _escape_like(
    value: str,  # Raw substring the caller wants matched literally
) -> str:  # Value with LIKE wildcards escaped (ESCAPE '\\')
    """Escape `%` / `_` / `\\` so `contains` matches literally, never as wildcards."""
    return value.replace("\\", "\\\\").replace("%", "\\%").replace("_", "\\_")

In [ ]:
#| export
def _predicate_sql(
    pred: PropertyPredicate,  # One typed property predicate
    props_col: str,  # SQL expression for the properties JSON column (e.g. "n.properties")
) -> Tuple[str, List[Any]]:  # (SQL fragment, bound params)
    """Compile one property predicate to a parameterized SQL fragment."""
    col = f"json_extract({props_col}, {_json_path(pred.prop)})"
    op = pred.op
    if op == "eq":
        return f"{col} = ?", [pred.value]
    if op == "ne":
        return f"{col} != ?", [pred.value]
    if op == "lt":
        return f"{col} < ?", [pred.value]
    if op == "le":
        return f"{col} <= ?", [pred.value]
    if op == "gt":
        return f"{col} > ?", [pred.value]
    if op == "ge":
        return f"{col} >= ?", [pred.value]
    if op == "in":
        if not isinstance(pred.value, (list, tuple)):
            raise CapabilityInputError(
                f"Op 'in' requires a list value (got {type(pred.value).__name__})",
                fields_invalid=["value"],
            )
        return (f"{col} IN (SELECT value FROM json_each(?))",
                [json.dumps(list(pred.value))])
    if op == "contains":
        if not isinstance(pred.value, str):
            raise CapabilityInputError(
                f"Op 'contains' requires a string value (got {type(pred.value).__name__})",
                fields_invalid=["value"],
            )
        # Defined case-INSENSITIVE (P12). Both sides through SQLite's lower()
        # (ASCII folding); wildcards escaped so the match is literal.
        return (f"lower({col}) LIKE '%' || lower(?) || '%' ESCAPE '\\'",
                [_escape_like(pred.value)])
    if op == "is_null":
        return f"{col} IS NULL", []
    if op == "not_null":
        return f"{col} IS NOT NULL", []
    raise CapabilityInputError(
        f"Predicate op {pred.op!r} not supported by the sqlite translation",
        fields_invalid=["op"],
    )

In [ ]:
#| export
def _source_match_sql(
    sp: SourcePredicate,  # Provenance match (content-hash-primary per CR-19)
    sources_expr: str,  # SQL expression for the sources JSON column (e.g. "n.sources")
) -> Tuple[str, List[Any]]:  # (EXISTS fragment, bound params)
    """Compile a source predicate to an EXISTS over the sources array.

    Content-hash-primary (identity field = the stable query surface, C19).
    A locator constraint RAISES — unsupported in the typed translation
    (use `find_nodes_by_source` for locator equality; recurring need here
    is the promotion evidence the raw-escape posture wants recorded).
    """
    if sp.locator is not None:
        raise CapabilityInputError(
            "SourcePredicate.locator is not supported by the sqlite typed "
            "translation (content-hash-primary); use find_nodes_by_source",
            fields_invalid=["locator"],
        )
    return (f"EXISTS (SELECT 1 FROM json_each({sources_expr}) AS src "
            f"WHERE json_extract(src.value, '$.content_hash') = ?)",
            [sp.content_hash])

In [ ]:
#| export
def _relation_exists_sql(
    rel: RelationPredicate,  # One-hop relation constraint (+ far-end constraints)
    node_expr: str,  # SQL expression for the candidate node id (e.g. "n.id", "e.source_id")
) -> Tuple[str, List[Any]]:  # (EXISTS fragment, bound params)
    """Compile a relation predicate to a correlated EXISTS (no row multiplication).

    Far-end constraints (stage-4 promotions): `node_id` / `node_ids` pin the
    far node; `node_source` nests a provenance EXISTS on the far node.
    Subquery scoping keeps the fixed aliases (r / fn / src) collision-free.
    """
    params: List[Any] = [rel.relation_type]
    if rel.direction == "out":
        join_cond = f"r.source_id = {node_expr}"
        far = "r.target_id"
    elif rel.direction == "in":
        join_cond = f"r.target_id = {node_expr}"
        far = "r.source_id"
    else:  # both
        join_cond = f"(r.source_id = {node_expr} OR r.target_id = {node_expr})"
        far = (f"(CASE WHEN r.source_id = {node_expr} "
               f"THEN r.target_id ELSE r.source_id END)")
    conds = [join_cond, "r.relation_type = ?"]
    if rel.node_id is not None:
        conds.append(f"{far} = ?")
        params.append(rel.node_id)
    if rel.node_ids is not None:
        conds.append(f"{far} IN (SELECT value FROM json_each(?))")
        params.append(json.dumps(list(rel.node_ids)))
    if rel.node_source is not None:
        src_frag, src_params = _source_match_sql(rel.node_source, "fn.sources")
        conds.append(
            f"EXISTS (SELECT 1 FROM nodes AS fn WHERE fn.id = {far} AND {src_frag})")
        params.extend(src_params)
    return f"EXISTS (SELECT 1 FROM edges AS r WHERE {' AND '.join(conds)})", params

In [ ]:
#| export
NODE_FULL_COLUMNS = "n.id, n.label, n.properties, n.sources, n.created_at, n.updated_at"  # Matches _row_to_node order
EDGE_FULL_COLUMNS = "e.id, e.source_id, e.target_id, e.relation_type, e.properties, e.created_at, e.updated_at"  # Matches _row_to_edge order


def _order_limit_sql(
    query,  # NodeQuery or EdgeQuery (order_by / limit / offset fields)
    props_col: str,  # Properties column expression for ORDER BY paths
    params: List[Any],  # Bound-params list (appended in place)
) -> str:  # ORDER BY / LIMIT / OFFSET tail
    """Compile the shared ordering + paging tail."""
    tail = ""
    if query.order_by is not None:
        tail += (f" ORDER BY json_extract({props_col}, "
                 f"{_json_path(query.order_by.prop)})"
                 + (" DESC" if query.order_by.descending else " ASC"))
    if query.limit is not None or query.offset:
        tail += " LIMIT ?"
        params.append(query.limit if query.limit is not None else -1)
        if query.offset:
            tail += " OFFSET ?"
            params.append(query.offset)
    return tail

In [ ]:
#| export
def translate_node_query(
    q: NodeQuery,  # Typed node query
) -> Tuple[str, List[Any], str, Optional[List[str]]]:  # (sql, params, mode, row keys)
    """Translate a `NodeQuery` to parameterized SQLite SQL.

    mode: "count" | "rows" | "full" (count > project > full, mirroring the
    result DTO's exactly-one-populated contract). For "rows", the returned
    keys list zips against each cursor row ("id" always first; "label"
    projects structurally; "sources" projects as the raw JSON column for the
    caller to parse).
    """
    where: List[str] = []
    params: List[Any] = []
    if q.ids is not None:
        where.append("n.id IN (SELECT value FROM json_each(?))")
        params.append(json.dumps(list(q.ids)))
    if q.label is not None:
        where.append("n.label = ?")
        params.append(q.label)
    for pred in q.where:
        frag, ps = _predicate_sql(pred, "n.properties")
        where.append(frag)
        params.extend(ps)
    if q.source is not None:
        frag, ps = _source_match_sql(q.source, "n.sources")
        where.append(frag)
        params.extend(ps)
    if q.related is not None:
        frag, ps = _relation_exists_sql(q.related, "n.id")
        where.append(frag)
        params.extend(ps)
    where_sql = (" WHERE " + " AND ".join(where)) if where else ""

    if q.count:
        return (f"SELECT COUNT(*) FROM nodes AS n{where_sql}", params, "count", None)

    if q.project is not None:
        keys = ["id"]
        select_parts = ["n.id"]
        for name in q.project:
            if name == "id":
                continue
            if name == "label":
                select_parts.append("n.label")
                keys.append("label")
            elif name == "sources":
                select_parts.append("n.sources")
                keys.append("sources")
            else:
                select_parts.append(f"json_extract(n.properties, {_json_path(name)})")
                keys.append(name)
        tail = _order_limit_sql(q, "n.properties", params)
        return (f"SELECT {', '.join(select_parts)} FROM nodes AS n{where_sql}{tail}",
                params, "rows", keys)

    tail = _order_limit_sql(q, "n.properties", params)
    return (f"SELECT {NODE_FULL_COLUMNS} FROM nodes AS n{where_sql}{tail}",
            params, "full", None)

In [ ]:
#| export
def translate_edge_query(
    q: EdgeQuery,  # Typed edge query
) -> Tuple[str, List[Any], str, Optional[List[str]]]:  # (sql, params, mode, row keys)
    """Translate an `EdgeQuery` to parameterized SQLite SQL.

    Same mode contract as `translate_node_query`. Projected rows always carry
    `id`/`source_id`/`target_id`; `relation_type` projects structurally.
    Endpoint constraints (`source_related`/`target_related` — the D13
    NEXT-chain count) compile to correlated EXISTS on the endpoint node.
    """
    where: List[str] = []
    params: List[Any] = []
    if q.ids is not None:
        where.append("e.id IN (SELECT value FROM json_each(?))")
        params.append(json.dumps(list(q.ids)))
    if q.relation_type is not None:
        where.append("e.relation_type = ?")
        params.append(q.relation_type)
    if q.source_id is not None:
        where.append("e.source_id = ?")
        params.append(q.source_id)
    if q.target_id is not None:
        where.append("e.target_id = ?")
        params.append(q.target_id)
    if q.source_ids is not None:
        where.append("e.source_id IN (SELECT value FROM json_each(?))")
        params.append(json.dumps(list(q.source_ids)))
    if q.target_ids is not None:
        where.append("e.target_id IN (SELECT value FROM json_each(?))")
        params.append(json.dumps(list(q.target_ids)))
    if q.source_related is not None:
        frag, ps = _relation_exists_sql(q.source_related, "e.source_id")
        where.append(frag)
        params.extend(ps)
    if q.target_related is not None:
        frag, ps = _relation_exists_sql(q.target_related, "e.target_id")
        where.append(frag)
        params.extend(ps)
    for pred in q.where:
        frag, ps = _predicate_sql(pred, "e.properties")
        where.append(frag)
        params.extend(ps)
    where_sql = (" WHERE " + " AND ".join(where)) if where else ""

    if q.count:
        return (f"SELECT COUNT(*) FROM edges AS e{where_sql}", params, "count", None)

    if q.project is not None:
        keys = ["id", "source_id", "target_id"]
        select_parts = ["e.id", "e.source_id", "e.target_id"]
        for name in q.project:
            if name in ("id", "source_id", "target_id"):
                continue
            if name == "relation_type":
                select_parts.append("e.relation_type")
                keys.append("relation_type")
            else:
                select_parts.append(f"json_extract(e.properties, {_json_path(name)})")
                keys.append(name)
        tail = _order_limit_sql(q, "e.properties", params)
        return (f"SELECT {', '.join(select_parts)} FROM edges AS e{where_sql}{tail}",
                params, "rows", keys)

    tail = _order_limit_sql(q, "e.properties", params)
    return (f"SELECT {EDGE_FULL_COLUMNS} FROM edges AS e{where_sql}{tail}",
            params, "full", None)

In [ ]:
# Behavior tests against the PRODUCTION schema in an in-memory DB — the
# translation contract is verified by what it returns, not by SQL strings.
import sqlite3
from cjm_context_graph_primitives.query import OrderBy

con = sqlite3.connect(":memory:")
con.execute("""CREATE TABLE nodes (
    id TEXT PRIMARY KEY, label TEXT NOT NULL, properties JSON, sources JSON,
    created_at REAL, updated_at REAL)""")
con.execute("""CREATE TABLE edges (
    id TEXT PRIMARY KEY, source_id TEXT NOT NULL, target_id TEXT NOT NULL,
    relation_type TEXT NOT NULL, properties JSON, created_at REAL, updated_at REAL)""")


def _node(id, label, props=None, sources=None):
    con.execute("INSERT INTO nodes VALUES (?,?,?,?,1.0,1.0)",
                (id, label, json.dumps(props or {}), json.dumps(sources or [])))


def _edge(id, src, tgt, rel, props=None):
    con.execute("INSERT INTO edges VALUES (?,?,?,?,?,1.0,1.0)",
                (id, src, tgt, rel, json.dumps(props or {})))


def run(q):
    if isinstance(q, NodeQuery):
        sql, params, mode, keys = translate_node_query(q)
    else:
        sql, params, mode, keys = translate_edge_query(q)
    cur = con.execute(sql, params)
    if mode == "count":
        return cur.fetchone()[0]
    rows = cur.fetchall()
    if mode == "rows":
        return [dict(zip(keys, r)) for r in rows]
    return rows


_node("doc-1", "Document", {"title": "Ep"})
_node("s0", "Segment", {"index": 0, "text": "It's 100% real_deal", "start_time": 0.0})
_node("s1", "Segment", {"index": 1, "text": "hello world", "start_time": 1.0},
      sources=[{"locator": {"kind": "file", "path": "/m.json"},
                "content_hash": "sha256:ab", "slice": None}])
_node("s2", "Segment", {"index": 2, "text": ""})
_node("s3", "Segment", {"index": 3})  # text key absent -> json_extract NULL
_node("c1", "Correction", {"correction_type": "text_content",
                           "payload": {"document_id": "doc-1", "segment_id": "s1"}})
_node("c2", "Correction", {"correction_type": "text_content",
                           "payload": {"segment_id": "s1"}})
_node("other", "Segment", {"index": 0, "text": "other doc"})  # NOT part of doc-1
for s in ("s0", "s1", "s2", "s3"):
    _edge(f"po-{s}", s, "doc-1", "PART_OF")
_edge("sw", "doc-1", "s0", "STARTS_WITH")
_edge("n0", "s0", "s1", "NEXT")
_edge("n1", "s1", "s2", "NEXT")
_edge("nx", "other", "other", "NEXT")  # NEXT edge OUTSIDE doc-1
_edge("co", "c1", "s1", "CORRECTS")
_edge("co2", "c2", "s1", "CORRECTS")
_edge("sup", "c2", "c1", "SUPERSEDES")
_edge("rev", "sess-1", "s0", "REVIEWED", {"decision": "corrected"})

# 1. C2 spine read: ordered projection with structural id + parsed-able sources
rows = run(NodeQuery(label="Segment",
                     related=RelationPredicate("PART_OF", node_id="doc-1"),
                     order_by=OrderBy(prop="index"),
                     project=["index", "text", "sources"]))
assert [r["index"] for r in rows] == [0, 1, 2, 3]
assert rows[0]["id"] == "s0" and json.loads(rows[1]["sources"])[0]["content_hash"] == "sha256:ab"

# 2. empty-filter OR case = two server-side counts (eq '' + is_null)
empty_eq = run(NodeQuery(label="Segment", where=[PropertyPredicate("text", "eq", "")],
                         related=RelationPredicate("PART_OF", node_id="doc-1"), count=True))
empty_null = run(NodeQuery(label="Segment", where=[PropertyPredicate("text", "is_null")],
                           related=RelationPredicate("PART_OF", node_id="doc-1"), count=True))
assert (empty_eq, empty_null) == (1, 1)

# 3. hash-cache promotion: Corrections whose CORRECTS target carries the hash
rows = run(NodeQuery(label="Correction",
                     related=RelationPredicate("CORRECTS", node_source=SourcePredicate(content_hash="sha256:ab"))))
assert sorted(r[0] for r in rows) == ["c1", "c2"]

# 4. C17 batch far end
rows = run(NodeQuery(label="Correction",
                     related=RelationPredicate("CORRECTS", node_ids=["s1", "nope"])))
assert sorted(r[0] for r in rows) == ["c1", "c2"]

# 5. D13 NEXT-chain count scoped to the document (excludes the outside edge)
n = run(EdgeQuery(relation_type="NEXT",
                  source_related=RelationPredicate("PART_OF", node_id="doc-1"), count=True))
assert n == 2
assert run(EdgeQuery(relation_type="NEXT", count=True)) == 3  # unscoped sees all

# 6. superseded-set via target id-list + structural projection
rows = run(EdgeQuery(relation_type="SUPERSEDES", target_ids=["c1", "zz"], project=[]))
assert [r["target_id"] for r in rows] == ["c1"]

# 7. dotted property path
rows = run(NodeQuery(label="Correction",
                     where=[PropertyPredicate("payload.document_id", "eq", "doc-1")]))
assert [r[0] for r in rows] == ["c1"]

# 8. REVIEWED markers: edge projection with property
rows = run(EdgeQuery(relation_type="REVIEWED", source_id="sess-1", project=["decision"]))
assert rows == [{"id": "rev", "source_id": "sess-1", "target_id": "s0",
                 "decision": "corrected"}]

# 9. contains: case-insensitive + literal wildcards (adversarial content)
hit = lambda needle: [r[0] for r in run(NodeQuery(label="Segment",
    where=[PropertyPredicate("text", "contains", needle)]))]
assert hit("IT'S") == ["s0"]          # case-insensitive, apostrophe-safe (bound param)
assert hit("100% real") == ["s0"]      # literal %
assert hit("l_dea") == ["s0"]          # literal _
assert hit("1%l") == []                # would match via wildcard if % unescaped
assert hit("Hello W") == ["s1"]

# 10. ne / order desc / limit (P12 Q5 shape) + paging
rows = run(NodeQuery(label="Segment", where=[PropertyPredicate("text", "ne", "")],
                     related=RelationPredicate("PART_OF", node_id="doc-1"),
                     order_by=OrderBy(prop="index", descending=True), limit=1))
assert rows[0][0] == "s1"  # s3 (null) + s2 ('') excluded; highest index with text
rows = run(NodeQuery(label="Segment", related=RelationPredicate("PART_OF", node_id="doc-1"),
                     order_by=OrderBy(prop="index"), limit=2, offset=1))
assert [r[0] for r in rows] == ["s1", "s2"]

# 11. in-op + ids batch + top-level source predicate
rows = run(NodeQuery(label="Segment", where=[PropertyPredicate("index", "in", [0, 2])],
                     related=RelationPredicate("PART_OF", node_id="doc-1")))
assert sorted(r[0] for r in rows) == ["s0", "s2"]
rows = run(NodeQuery(ids=["s1", "c1"]))
assert sorted(r[0] for r in rows) == ["c1", "s1"]
rows = run(NodeQuery(source=SourcePredicate(content_hash="sha256:ab")))
assert [r[0] for r in rows] == ["s1"]

# 12. direction 'in' + 'both'
rows = run(NodeQuery(related=RelationPredicate("STARTS_WITH", direction="in", node_id="doc-1")))
assert [r[0] for r in rows] == ["s0"]
rows = run(NodeQuery(related=RelationPredicate("NEXT", direction="both", node_id="s1")))
assert sorted(r[0] for r in rows) == ["s0", "s2"]

# 13. unsupported / invalid raise loudly
for bad in (lambda: translate_node_query(NodeQuery(source=SourcePredicate(
                content_hash="sha256:ab",
                locator=__import__("cjm_context_graph_primitives.locators",
                                   fromlist=["FileRef"]).FileRef("/x")))),
            lambda: translate_node_query(NodeQuery(where=[PropertyPredicate("a", "in", "notalist")])),
            lambda: translate_node_query(NodeQuery(where=[PropertyPredicate("text; DROP", "eq", "x")])),
            lambda: translate_node_query(NodeQuery(order_by=OrderBy(prop="a'b")))):
    try:
        bad()
        raise AssertionError("should have raised")
    except CapabilityInputError:
        pass

con.close()